# 필수 검색과 선택 도구를 한 서비스에서 연결하기

서비스에 들어온 모든 요청을 문서 검색에 보내면 인사까지 규정 답변이 된다. 반대로 검색을 건너뛰면 조항을 확인해야 할 문의에 모델이 임의로 답할 수 있다.

### 노트북 실행 준비

Python 3.12 커널에서 위쪽 셀부터 실행하세요. 패키지가 없는 커널에서는 새 코드 셀에 다음 명령을 실행한 뒤 예제로 이동하세요.

%pip install "langgraph>=1,<2" "langchain-core>=1,<2"

## 검색기를 어디에 놓을까?

근거를 반드시 제시해야 하는 규정 문의는 검색 노드를 거치게 만든다. 업무 상태 조회 같은 요청에는 접수 저장소가 필요하다. 둘을 모두 문서 검색 도구로 처리하려 하면 필요한 입력도 반환값도 달라진다.

검색기 함수가 문서 ID와 문장을 함께 반환하도록 약속해 두면 고정 노드와 검색 도구가 같은 자료를 읽을 수 있다. 결과가 달라졌을 때 검색 구현과 호출 경로 중 어디를 바꿨는지도 구분하기 쉽다.

도구 호출 메시지의 ID는 실행 요청을 가리킨다. 문서 ID는 근거를 가리킨다. 한 필드에 두 값을 섞으면 두 번의 검색 가운데 어느 결과로 답했는지 되짚기 어렵다.

## 같은 검색 함수를 도구로 열기

LangChain 도구를 호출 요청 전체로 실행하면 결과에는 요청 ID가 남는다. 반환 본문에는 별도로 근거 ID를 둔다.

In [1]:
from langchain_core.tools import tool
def retrieve(topic):
    return {"연장":"R2 · 연장은 한 번 가능합니다."}.get(topic, "근거 없음")

@tool
def find_rule(topic: str) -> str:
    """질문 주제에 해당하는 규정 문장을 찾는다."""
    return retrieve(topic)

call = {"name":"find_rule", "args":{"topic":"연장"}, "id":"req-7", "type":"tool_call"}
reply = find_rule.invoke(call)
print(reply.tool_call_id, reply.content)

req-7 R2 · 연장은 한 번 가능합니다.


req-7은 호출과 결과를 맞추는 값이다. 학생이 원문을 확인할 때 따라갈 값은 R2다. 이 셀에서는 도구를 직접 호출했으므로 모델이 도구를 골랐는지는 시험하지 않았다.

### 근거 없음에서 답변을 보류하기

같은 도구 이름에 다른 인자와 새 요청 ID를 보낸다. 직전 호출의 R2가 재사용되지 않는지 확인한다.

In [2]:
missing = dict(call, id="req-8", args={"topic":"주차"})

In [3]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>result = find_rule.invoke(missing)
print(result.tool_call_id, result.content)</code></pre><pre>req-8 근거 없음
</pre><p>req-8의 결과에는 근거가 없다. 앞서 받은 ToolMessage를 새 호출에 붙이면 저장된 문서가 틀려도 답 자체는 그럴듯해질 수 있다.</p></details>

## 도구를 언제 부를지 그래프에서 관찰하기

규정 문의와 인사말을 한 입구에서 처리한다. 입력의 intent는 미리 붙인 시험값이다. 모델 분류 성능과 검색 경로 시험을 분리해 두면 실패 위치가 보인다.

In [4]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
class Query(TypedDict, total=False):
    intent: str
    topic: str
    reply: str
g = StateGraph(Query)
g.add_node("lookup", lambda s: {"reply":retrieve(s["topic"])})
g.add_node("hello", lambda s: {"reply":"안녕하세요."})
g.add_conditional_edges(START, lambda s:s["intent"], {"규정":"lookup","인사":"hello"})
g.add_edge("lookup", END)
g.add_edge("hello", END)
router = g.compile()
print(list(router.stream({"intent":"인사","topic":""}, stream_mode="updates")))

[{'hello': {'reply': '안녕하세요.'}}]


인사 입력에서는 hello만 실행된다. 단순히 마지막 답에 출처가 없다는 사실과 실제 검색 노드가 생략됐다는 사실은 서로 다른 관찰이다.

### 근거 없는 규정 문의의 경로 조사

규정이라는 의도는 맞지만 문서에 주차 규정이 없다. 검색을 건너뛴 결과와 검색은 했지만 비어 있는 결과를 구별하도록 실행 기록을 남긴다.

In [5]:
case = {"intent":"규정", "topic":"주차"}

In [6]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>print(list(router.stream(case, stream_mode=&quot;updates&quot;)))</code></pre><pre>[{&#x27;lookup&#x27;: {&#x27;reply&#x27;: &#x27;근거 없음&#x27;}}]
</pre><p>lookup 노드가 실행됐고 본문은 근거 없음이다. 이 기록을 인사 경로와 비교하면 어느 단계의 정책을 고쳐야 할지 판단할 수 있다.</p></details>

## 다른 입력에 적용하기

같은 세 질문을 고정 검색 그래프와 도구 선택 그래프에 넣어 본다. 인사에는 검색이 필요 없고, 규정 문의에는 출처가 있어야 하며, 미등록 규정에는 보류가 필요하다.

답변·검색 횟수·호출 ID·문서 ID를 적는다. 검색 횟수가 줄어든 대신 필수 근거를 빠뜨린 질문이 있다면 그 입력을 회귀 사례로 보관한다.

## 실제 모델이 규정 검색을 선택하는 과정

검색 요청을 입력에 미리 넣지 않고 사용자 질문을 보낸다. 반환된 도구 이름과 인자로 검색을 실행한다.

수업 커널에 langchain-openai와 langchain을 설치하고 OPENAI_API_KEY를 환경 변수에 설정한다. 호출에는 API 요금이 발생한다.

In [ ]:
import os
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model=os.getenv("OPENAI_MODEL") or "gpt-4o-mini", timeout=45, max_retries=1)
from langchain.agents import create_agent
agent = create_agent(model, tools=[find_rule], system_prompt="규정을 물으면 find_rule로 주제 키워드를 검색하고 근거와 함께 답하세요.")
history = agent.invoke({"messages":[("user","연장 규정이 궁금합니다.")]})["messages"]
for message in history:
    print(message.type, getattr(message,"tool_calls",None) or message.content)

인사와 규정 질문을 각각 보내 어느 경우에 실제 조회가 발생하는지 비교한다.